<a href="https://colab.research.google.com/github/TahaShan16/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TahaShan16/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring. I would frame this as a ranking or scoring task, not mainly as classification. The goal is to rank existing pages by which ones should be reviewed first by a content or SEO team. The model output would be a priority score for each page, and the highest-scoring pages would become the review queue.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

A useful target would be an observed future outcome, such as whether a page later lost meaningful impressions or needed a successful refresh. In the starter dataset, I do not have future outcomes yet, so I will sketch a proxy target for now. A possible proxy is whether a visible page is declining, using trend direction and recent impressions. This is only a temporary proxy, not a final truth label.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The success metric should match the review queue. I would use Precision@50 because a content team may only have time to review the top 50 recommended pages. Precision@50 asks: among the 50 pages the system recommends first, how many actually match the target or proxy for a useful review? This is better than plain accuracy because most pages may not need action, and the top of the queue matters most.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one page. In the lane slice, I kept pages with at least 500 impressions in the last 90 days, which gives 16,726 rows. The temporary proxy target is proxy_needs_review: 1 means the page is either declining or has not been updated recently, and 0 means it does not match that proxy. This proxy marks 9,962 rows as positive, or about 59.6% of the lane slice. This is only a sketch target for framing; a stronger later target should use an observed future outcome.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os
import requests

local_file_path = "data/raw/content_refresh_anonymized.csv"
github_raw_url = "https://raw.githubusercontent.com/TahaShan16/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

if not os.path.exists(local_file_path):
    response = requests.get(github_raw_url)
    response.raise_for_status()
    with open(local_file_path, "wb") as f:
        f.write(response.content)

df = pd.read_csv(local_file_path)

lane_df = df[df["impressions_90d"] >= 500].copy()

lane_df["proxy_needs_review"] = (
    (lane_df["trend_direction"].str.lower() == "down") |
    (lane_df["days_since_last_update"] >= 180)
).astype(int)

possible_page_cols = ["page_id", "url", "page_url", "landing_page", "slug"]
page_col = next((c for c in possible_page_cols if c in lane_df.columns), None)

display_cols = [
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "days_since_last_update",
    "proxy_needs_review",
]

if page_col:
    display_cols = [page_col] + display_cols

unit_view = lane_df[display_cols].head(10)

print("Unit of analysis: one row = one page")
print("Page identifier column:", page_col if page_col else "No separate page ID column found")
print("Lane slice rows:", len(lane_df))
print("Proxy positive rows:", lane_df["proxy_needs_review"].sum())
print("Proxy positive rate:", round(lane_df["proxy_needs_review"].mean(), 3))

unit_view

print("Unit of analysis: one row = one page")
print("Lane slice rows:", len(lane_df))
print("Proxy positive rows:", lane_df["proxy_needs_review"].sum())
print("Proxy positive rate:", round(lane_df["proxy_needs_review"].mean(), 3))

unit_view


Unit of analysis: one row = one page
Page identifier column: No separate page ID column found
Lane slice rows: 16726
Proxy positive rows: 9962
Proxy positive rate: 0.596
Unit of analysis: one row = one page
Lane slice rows: 16726
Proxy positive rows: 9962
Proxy positive rate: 0.596


,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,days_since_last_update,proxy_needs_review
0,client_f369cb89fc,keyword article,3803,29,0.76,10.6,down,20,1
1,client_4e07408562,keyword article,15320,7,0.05,20.3,down,25,1
2,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,down,20,1
3,client_19581e27de,keyword article,11751,58,0.49,6.2,stable,22,0
4,client_3fdba35f04,keyword article,19140,24,0.13,44.0,down,14,1
5,client_f369cb89fc,keyword article,3970,1,0.03,8.5,down,20,1
7,client_19581e27de,keyword article,1724,1,0.06,21.2,stable,22,0
8,client_6208ef0f77,keyword article,32574,29,0.09,46.0,down,20,1
9,client_19581e27de,keyword article,1240,2,0.16,4.9,down,104,1
10,client_19581e27de,keyword article,20919,324,1.55,2.2,stable,104,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as “review every declining page” would be easy to understand, but it may be too blunt. It would not rank pages by business value, visibility, CTR, position, content type, or how recently the page was updated. A scoring or ranking model can combine several signals and sort pages into a practical review queue. ML is useful here if it helps decide which pages deserve attention first, not if it only repeats a rule that humans already wrote.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.